In [1]:
from pathlib import Path
import glob
import re

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# ==============================
# Project paths
# ==============================

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

NTL_GHSL_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "vnp46a2"
    / "ghsl_masked_ntl_samar_leyte"
)

NGCP_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "ngcp"
    / "NGCP_Hourly_Demand.csv"
)

NGCP_PROCESSED_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "ngcp"
    / "NGCP_1AM_Demand_processed.csv"
)

GAP_FILLED_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "vnp46a2"
    / "ghsl_masked_gapfilled_samar_leyte"
)

OUTPUT_MAIN_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "main"
OUTPUT_SUPP_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "supplementary"
OUTPUT_MAIN_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "main"
OUTPUT_SUPP_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "supplementary"

for out_dir in [
    OUTPUT_MAIN_FIG_DIR,
    OUTPUT_SUPP_FIG_DIR,
    OUTPUT_MAIN_TABLE_DIR,
    OUTPUT_SUPP_TABLE_DIR,
]:
    out_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# ==============================
# Analysis settings
# ==============================

EVENT_DATE = pd.Timestamp("2013-11-08")

BASELINE_START = EVENT_DATE - pd.Timedelta(days=90)
BASELINE_END = EVENT_DATE - pd.Timedelta(days=8)

ASSESSMENT_START = BASELINE_START
ASSESSMENT_END = EVENT_DATE + pd.Timedelta(days=180)

GROUP = "23-30"
GROUP_CODE = "G2"
GROUP_LABEL = "Dense Urban"

VALUE_COL = "Mean_by_sum"
VALID_THRESHOLD = 60
IMPACT_WINDOW_DAYS = 21

print(f"Baseline: {BASELINE_START.date()} to {BASELINE_END.date()}")
print(f"Assessment: {ASSESSMENT_START.date()} to {ASSESSMENT_END.date()}")

Baseline: 2013-08-10 to 2013-10-31
Assessment: 2013-08-10 to 2014-05-07


In [ ]:
# ==============================
# Analysis settings
# ==============================

EVENT_DATE = pd.Timestamp("2013-11-08")

BASELINE_START = EVENT_DATE - pd.Timedelta(days=90)
BASELINE_END = EVENT_DATE - pd.Timedelta(days=8)

ASSESSMENT_START = BASELINE_START
ASSESSMENT_END = EVENT_DATE + pd.Timedelta(days=180)

IMPACT_END = EVENT_DATE + pd.Timedelta(days=21)

VALUE_COL = "Mean_by_sum"
VALID_THRESHOLD = 10
BIN_DAYS = 14
MAX_CONFIRMATION_GAP = 14
NGCP_SUSTAINED_DAYS = 7

GROUP_META = {
    "30": {
        "code": "G1",
        "label": "Urban Core",
        "color": "#d73027",
    },
    "23-30": {
        "code": "G2",
        "label": "Dense Urban",
        "color": "#8c2d04",
    },
    "22-30": {
        "code": "G3",
        "label": "Core & Inner",
        "color": "#e08214",
    },
}

GROUP_ORDER = ["30", "23-30", "22-30"]

PHASES = {
    "Immediate baseline": (
        EVENT_DATE - pd.Timedelta(days=90),
        EVENT_DATE - pd.Timedelta(days=8),
    ),
    "Core shock": (
        EVENT_DATE - pd.Timedelta(days=7),
        EVENT_DATE + pd.Timedelta(days=21),
    ),
    "Early recovery": (
        EVENT_DATE + pd.Timedelta(days=22),
        EVENT_DATE + pd.Timedelta(days=90),
    ),
    "Medium recovery": (
        EVENT_DATE + pd.Timedelta(days=91),
        EVENT_DATE + pd.Timedelta(days=180),
    ),
}

In [72]:
# ==============================
# Plot helpers
# ==============================

def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(
        int(hex_color[i:i + 2], 16)
        for i in (0, 2, 4)
    )
    return f"rgba({r},{g},{b},{alpha})"


def add_haiyan_marker(
    fig,
    row=None,
    col=None,
    annotation=True,
    annotation_y=0.95,
):
    marker = dict(
        x=EVENT_DATE.to_pydatetime(),
        line_dash="dash",
        line_color="#1f77b4",
        line_width=2,
    )

    if row is None:
        fig.add_vline(**marker)
    else:
        fig.add_vline(**marker, row=row, col=col)

    if annotation:
        fig.add_annotation(
            x=EVENT_DATE,
            y=annotation_y,
            yref="paper",
            text="Haiyan · 8 Nov 2013",
            showarrow=False,
            font=dict(size=15, color="#1f77b4"),
            bgcolor="rgba(255,255,255,0.88)",
            bordercolor="#1f77b4",
            borderwidth=1,
            xanchor="left",
        )

    return fig


def apply_figure_style(
    fig,
    width=1250,
    height=650,
    top=100,
    right=50,
):
    fig.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=width,
        height=height,
        margin=dict(
            t=top,
            b=75,
            l=85,
            r=right,
        ),
        font=dict(size=16, color="black"),
        legend=dict(
            orientation="h",
            x=0.7,
            xanchor="center",
            y=0.90,
            yanchor="bottom",
            font=dict(size=14),
        ),
    )

    fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
    )

    fig.update_yaxes(
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
    )

    return fig


def export_figure(fig, output_path):
    output_path = Path(output_path)

    fig.write_html(
        str(output_path.with_suffix(".html")),
        include_plotlyjs="cdn",
        full_html=True,
    )

    print(f"Exported: {output_path.with_suffix('.html')}")

In [44]:
# ==============================
# Load G1–G3 DNB-BRDF statistics
# ==============================

csv_files = sorted(
    glob.glob(str(NTL_GHSL_DIR / "*.csv"))
)

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in: {NTL_GHSL_DIR}"
    )

df_list = []

for file in csv_files:
    file_path = Path(file)

    group = (
        file_path.name
        .replace("EasternVisayas_DNBBRDF_", "")
        .replace("_stats.csv", "")
    )

    if group not in GROUP_ORDER:
        continue

    temp = pd.read_csv(file_path)

    if "date" not in temp.columns:
        temp["date"] = (
            temp["system:index"]
            .astype(str)
            .str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
            .str.replace("_", "-", regex=False)
        )

    temp["date"] = pd.to_datetime(
        temp["date"],
        errors="coerce",
    )

    numeric_cols = [
        VALUE_COL,
        "Valid_px",
        "NTL_p05",
        "NTL_p25",
        "NTL_median",
        "NTL_p75",
        "NTL_p95",
    ]

    for col in numeric_cols:
        temp[col] = pd.to_numeric(
            temp[col],
            errors="coerce",
        )

    temp["Group"] = group
    df_list.append(temp)

df_all = (
    pd.concat(df_list, ignore_index=True)
    .dropna(subset=["date"])
    .sort_values(["Group", "date"])
    .reset_index(drop=True)
)

prepared_groups = []

for group in GROUP_ORDER:
    gdf = df_all[
        df_all["Group"] == group
    ].copy()

    total_px = gdf["Valid_px"].max()

    gdf["Total_px"] = total_px
    gdf["SC_pct"] = (
        gdf["Valid_px"] / total_px * 100
    )

    valid_values = gdf.loc[
        (gdf["Valid_px"] > 0)
        & gdf[VALUE_COL].notna(),
        VALUE_COL,
    ]

    p00, p95 = np.nanpercentile(
        valid_values,
        [0, 95],
    )

    gdf["NTL_value"] = gdf[
        VALUE_COL
    ].clip(p00, p95)

    gdf["Qualified"] = (
        (gdf["SC_pct"] >= VALID_THRESHOLD)
        & gdf["NTL_value"].notna()
    )

    prepared_groups.append(gdf)

df_panel = (
    pd.concat(prepared_groups, ignore_index=True)
    .sort_values(["Group", "date"])
    .reset_index(drop=True)
)

print(
    df_panel.groupby("Group").agg(
        Start=("date", "min"),
        End=("date", "max"),
        Total_px=("Total_px", "max"),
        Qualified_days=("Qualified", "sum"),
    )
)

           Start        End  Total_px  Qualified_days
Group                                                
22-30 2012-01-19 2025-07-21      2782            2728
23-30 2012-01-19 2025-07-21      1492            2693
30    2012-01-19 2025-07-21       449            2526


In [27]:
# ==============================
# Load NGCP 1 AM demand
# ==============================

if NGCP_PATH.exists():
    df_ngcp_raw = pd.read_csv(
        NGCP_PATH,
        skiprows=2,
    )

    df_ngcp_raw.columns = [
        "Date",
        "Coincident Peak",
        "Hour No",
    ] + [str(hour) for hour in range(1, 25)]

    df_ngcp = (
        df_ngcp_raw[["Date", "1"]]
        .rename(columns={"1": "Grid_1AM_MW"})
        .copy()
    )

    df_ngcp["Date"] = pd.to_datetime(
        df_ngcp["Date"],
        errors="coerce",
        dayfirst=True,
    )

elif NGCP_PROCESSED_PATH.exists():
    df_ngcp = pd.read_csv(
        NGCP_PROCESSED_PATH
    )

    df_ngcp["Date"] = pd.to_datetime(
        df_ngcp["Date"],
        errors="coerce",
    )

else:
    raise FileNotFoundError(
        "NGCP data were not found."
    )

df_ngcp["Grid_1AM_MW"] = pd.to_numeric(
    df_ngcp["Grid_1AM_MW"],
    errors="coerce",
)

df_ngcp = (
    df_ngcp
    .dropna(subset=["Date", "Grid_1AM_MW"])
    .sort_values("Date")
    .reset_index(drop=True)
)

print(f"NGCP rows: {len(df_ngcp):,}")
print(
    f"Date range: {df_ngcp['Date'].min().date()} "
    f"to {df_ngcp['Date'].max().date()}"
)

NGCP rows: 4,383
Date range: 2013-01-01 to 2024-12-31


In [28]:
def maximum_observation_gap(dates):
    dates = (
        pd.Series(pd.to_datetime(dates))
        .dropna()
        .sort_values()
    )

    if len(dates) < 2:
        return np.nan

    return dates.diff().dt.days.max()


support_rows = []

for group in GROUP_ORDER:
    meta = GROUP_META[group]

    gdf = df_panel[
        df_panel["Group"] == group
    ].copy()

    for phase, (phase_start, phase_end) in PHASES.items():
        phase_all = gdf[
            (gdf["date"] >= phase_start)
            & (gdf["date"] <= phase_end)
        ].copy()

        phase_valid = phase_all[
            phase_all["Qualified"]
        ].copy()

        phase_days = (
            phase_end - phase_start
        ).days + 1

        support_rows.append({
            "Group": group,
            "Code": meta["code"],
            "Label": meta["label"],
            "Phase": phase,
            "Phase_days": phase_days,
            "Qualified_days": len(phase_valid),
            "Temporal_support_pct": (
                len(phase_valid)
                / phase_days
                * 100
            ),
            "Median_SC_all_days": (
                phase_all["SC_pct"].median()
            ),
            "Median_SC_qualified_days": (
                phase_valid["SC_pct"].median()
            ),
            "Maximum_gap_days": (
                maximum_observation_gap(
                    phase_valid["date"]
                )
            ),
        })

support_df = pd.DataFrame(support_rows)

support_df.to_csv(
    OUTPUT_MAIN_TABLE_DIR
    / "table_01_haiyan_observation_support_g1_g3.csv",
    index=False,
)

support_df.round(2)

,Group,Code,Label,Phase,Phase_days,Qualified_days,Temporal_support_pct,Median_SC_all_days,Median_SC_qualified_days,Maximum_gap_days
0,30,G1,Urban Core,Immediate baseline,83,25,30.12,0.00,70.16,14.0
1,30,G1,Urban Core,Core shock,29,15,51.72,17.82,44.99,7.0
2,30,G1,Urban Core,Early recovery,69,36,52.17,10.02,64.14,19.0
3,30,G1,Urban Core,Medium recovery,90,67,74.44,40.87,65.26,7.0
4,23-30,G2,Dense Urban,Immediate baseline,83,25,30.12,0.40,47.45,14.0
5,23-30,G2,Dense Urban,Core shock,29,17,58.62,18.16,51.61,6.0
6,23-30,G2,Dense Urban,Early recovery,69,37,53.62,15.01,55.90,17.0
7,23-30,G2,Dense Urban,Medium recovery,90,70,77.78,43.73,60.36,7.0
8,22-30,G3,Core & Inner,Immediate baseline,83,26,31.33,0.79,47.93,9.0
9,22-30,G3,Core & Inner,Core shock,29,18,62.07,19.12,53.20,6.0


In [29]:
event_groups = {}
baseline_rows = []

for group in GROUP_ORDER:
    meta = GROUP_META[group]

    gdf = df_panel[
        (df_panel["Group"] == group)
        & (df_panel["date"] >= ASSESSMENT_START)
        & (df_panel["date"] <= ASSESSMENT_END)
        & df_panel["Qualified"]
    ].copy()

    baseline_values = gdf.loc[
        (gdf["date"] >= BASELINE_START)
        & (gdf["date"] <= BASELINE_END),
        "NTL_value",
    ]

    baseline = baseline_values.median()

    if not np.isfinite(baseline) or baseline <= 0:
        raise ValueError(
            f"Invalid baseline for {meta['code']}"
        )

    gdf["Pct_baseline"] = (
        gdf["NTL_value"]
        / baseline
        * 100
    )

    gdf["Days_since_event"] = (
        gdf["date"] - EVENT_DATE
    ).dt.days

    event_groups[group] = gdf

    baseline_rows.append({
        "Group": group,
        "Code": meta["code"],
        "Label": meta["label"],
        "Baseline": baseline,
        "Baseline_observations": len(
            baseline_values
        ),
    })

df_ngcp_event = df_ngcp[
    (df_ngcp["Date"] >= ASSESSMENT_START)
    & (df_ngcp["Date"] <= ASSESSMENT_END)
].copy()

ngcp_baseline_values = df_ngcp_event.loc[
    (df_ngcp_event["Date"] >= BASELINE_START)
    & (df_ngcp_event["Date"] <= BASELINE_END),
    "Grid_1AM_MW",
]

ngcp_baseline = ngcp_baseline_values.median()

df_ngcp_event["Pct_baseline"] = (
    df_ngcp_event["Grid_1AM_MW"]
    / ngcp_baseline
    * 100
)

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.round(3)

,Group,Code,Label,Baseline,Baseline_observations
0,30,G1,Urban Core,3.673,25
1,23-30,G2,Dense Urban,1.936,25
2,22-30,G3,Core & Inner,1.189,26


In [46]:
daily_index = pd.date_range(
    ASSESSMENT_START,
    ASSESSMENT_END,
    freq="D",
)

heatmap_rows = []

for group in GROUP_ORDER:
    gdf = (
        df_panel[
            (df_panel["Group"] == group)
            & (
                df_panel["date"]
                >= ASSESSMENT_START
            )
            & (
                df_panel["date"]
                <= ASSESSMENT_END
            )
        ]
        .set_index("date")
        .reindex(daily_index)
    )

    heatmap_rows.append(
        gdf["SC_pct"]
        .fillna(0)
        .to_numpy()
    )

fig_observability = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.20, 0.80],
    vertical_spacing=0.1,
    subplot_titles=(
        "Daily spatial completeness",
        # "Reliability-qualified NTL observations and NGCP load",
    ),
)

fig_observability.add_trace(
    go.Heatmap(
        z=np.asarray(heatmap_rows),
        x=daily_index,
        y=[
            GROUP_META[group]["code"]
            for group in GROUP_ORDER
        ],
        colorscale="Greens",
        zmin=0,
        zmax=100,
        colorbar=dict(
            title="SC (%)",
            len=0.25,
            y=0.92,
            thickness=16,
        ),
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Group: %{y}<br>"
            "SC: %{z:.1f}%"
            "<extra></extra>"
        ),
        name="Spatial completeness",
    ),
    row=1,
    col=1,
)

for group in GROUP_ORDER:
    meta = GROUP_META[group]
    gdf = event_groups[group]

    fig_observability.add_trace(
        go.Scatter(
            x=gdf["date"],
            y=gdf["Pct_baseline"],
            mode="markers",
            marker=dict(
                size=8,
                color=meta["color"],
                line=dict(
                    color="white",
                    width=0.7,
                ),
            ),
            customdata=np.column_stack([
                gdf["SC_pct"],
                gdf["Valid_px"],
            ]),
            hovertemplate=(
                "Date: %{x|%Y-%m-%d}<br>"
                f"{meta['code']} NTL: "
                "%{y:.1f}% of baseline<br>"
                "SC: %{customdata[0]:.1f}%<br>"
                "Valid pixels: "
                "%{customdata[1]:,.0f}"
                "<extra></extra>"
            ),
            name=(
                f"{meta['code']}: "
                f"{meta['label']}"
            ),
        ),
        row=2,
        col=1,
    )

fig_observability.add_trace(
    go.Scatter(
        x=df_ngcp_event["Date"],
        y=df_ngcp_event["Pct_baseline"],
        mode="lines",
        line=dict(
            color="black",
            width=2.5,
        ),
        name="NGCP 1 AM load",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "NGCP: %{y:.1f}% of baseline"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

fig_observability.add_hline(
    y=100,
    line_dash="dot",
    line_color="grey",
    line_width=1.5,
    row=2,
    col=1,
)

add_haiyan_marker(
    fig_observability,
    row=1,
    col=1,
    annotation=False,
)

add_haiyan_marker(
    fig_observability,
    row=2,
    col=1,
    annotation=True,
)

fig_observability.update_layout(
    title=(
        "Haiyan Regional Observability and Functional Recovery"
        "<br><sup>"
        "G1–G3 DNB-BRDF, spatial completeness "
        f"≥ {VALID_THRESHOLD}%"
        "</sup>"
    )
)

fig_observability.update_yaxes(
    title_text="",
    row=1,
    col=1,
)

fig_observability.update_yaxes(
    title_text=(
        "Percentage of pre-Haiyan baseline (%)"
    ),
    rangemode="tozero",
    row=2,
    col=1,
)

fig_observability.update_xaxes(
    title_text="Date",
    row=2,
    col=1,
)

apply_figure_style(
    fig_observability,
    width=1300,
    height=700,
    top=80,
)

fig_observability.show()

export_figure(
    fig_observability,
    OUTPUT_MAIN_FIG_DIR
    / "fig_01_haiyan_observability_recovery_g1_g3",
)

Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/outputs/figures/main/fig_01_haiyan_observability_recovery_g1_g3.html


In [59]:
DISTRIBUTION_START = (
    EVENT_DATE - pd.Timedelta(days=30)
)

DISTRIBUTION_END = (
    EVENT_DATE + pd.Timedelta(days=90)
)

fig_percentiles = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[
        (
            f"{GROUP_META[group]['code']}: "
            f"{GROUP_META[group]['label']}"
        )
        for group in GROUP_ORDER
    ],
)

for row, group in enumerate(
    GROUP_ORDER,
    start=1,
):
    meta = GROUP_META[group]

    gdf = event_groups[group][
        (
            event_groups[group]["date"]
            >= DISTRIBUTION_START
        )
        & (
            event_groups[group]["date"]
            <= DISTRIBUTION_END
        )
    ].dropna(subset=[
        "NTL_p05",
        "NTL_p25",
        "NTL_median",
        "NTL_p75",
        "NTL_p95",
    ]).copy()

    # p05–p95 whisker
    fig_percentiles.add_trace(
        go.Scatter(
            x=gdf["date"],
            y=gdf["NTL_median"],
            mode="markers",
            marker=dict(
                size=1,
                color="rgba(0,0,0,0)",
            ),
            error_y=dict(
                type="data",
                symmetric=False,
                array=(
                    gdf["NTL_p95"]
                    - gdf["NTL_median"]
                ).clip(lower=0),
                arrayminus=(
                    gdf["NTL_median"]
                    - gdf["NTL_p05"]
                ).clip(lower=0),
                color=hex_to_rgba(
                    meta["color"],
                    0.28,
                ),
                thickness=1,
                width=0,
            ),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row,
        col=1,
    )

    # p25–p75 with median marker
    fig_percentiles.add_trace(
        go.Scatter(
            x=gdf["date"],
            y=gdf["NTL_median"],
            mode="markers",
            marker=dict(
                size=7,
                color=meta["color"],
                line=dict(
                    color="white",
                    width=0.7,
                ),
            ),
            error_y=dict(
                type="data",
                symmetric=False,
                array=(
                    gdf["NTL_p75"]
                    - gdf["NTL_median"]
                ).clip(lower=0),
                arrayminus=(
                    gdf["NTL_median"]
                    - gdf["NTL_p25"]
                ).clip(lower=0),
                color=meta["color"],
                thickness=5,
                width=0,
            ),
            customdata=np.column_stack([
                gdf["NTL_p05"],
                gdf["NTL_p25"],
                gdf["NTL_p75"],
                gdf["NTL_p95"],
                gdf["SC_pct"],
            ]),
            hovertemplate=(
                "Date: %{x|%Y-%m-%d}<br>"
                "Median: %{y:.3f}<br>"
                "p25–p75: "
                "%{customdata[1]:.3f}–"
                "%{customdata[2]:.3f}<br>"
                "p05–p95: "
                "%{customdata[0]:.3f}–"
                "%{customdata[3]:.3f}<br>"
                "SC: %{customdata[4]:.1f}%"
                "<extra></extra>"
            ),
            name=meta["code"],
            showlegend=False,
        ),
        row=row,
        col=1,
    )

    add_haiyan_marker(
        fig_percentiles,
        row=row,
        col=1,
        annotation=(row == 1),
    )

    fig_percentiles.update_yaxes(
        title_text="Radiance",
        rangemode="tozero",
        row=row,
        col=1,
    )

fig_percentiles.update_layout(
    title=(
        "Daily Spatial Distribution of Observed DNB-BRDF"
        "<br><sup>"
        "Thin: p05–p95 · Thick: p25–p75 · "
        "Marker: median"
        "</sup>"
    )
)

fig_percentiles.update_xaxes(
    title_text="Date",
    row=3,
    col=1,
)


apply_figure_style(
    fig_percentiles,
    width=1300,
    height=900,
    top=110,
)

# fig_percentiles.update_yaxes(type="log")

fig_percentiles.show()

export_figure(
    fig_percentiles,
    OUTPUT_SUPP_FIG_DIR
    / "fig_02_haiyan_daily_spatial_percentiles_g1_g3",
)

Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/outputs/figures/supplementary/fig_02_haiyan_daily_spatial_percentiles_g1_g3.html


In [60]:
def build_binned_profile(
    data,
    date_col,
    value_col,
    sc_col=None,
):
    columns = [date_col, value_col]

    if sc_col:
        columns.append(sc_col)

    temp = (
        data[columns]
        .dropna(subset=[date_col, value_col])
        .copy()
    )

    temp["Days_since_event"] = (
        temp[date_col] - EVENT_DATE
    ).dt.days

    temp["Bin_id"] = np.floor(
        temp["Days_since_event"] / BIN_DAYS
    ).astype(int)

    agg_dict = {
        "Median": (value_col, "median"),
        "P25": (
            value_col,
            lambda values: values.quantile(0.25),
        ),
        "P75": (
            value_col,
            lambda values: values.quantile(0.75),
        ),
        "N_obs": (value_col, "count"),
    }

    if sc_col:
        agg_dict["Median_SC"] = (
            sc_col,
            "median",
        )

    binned = (
        temp.groupby("Bin_id", as_index=False)
        .agg(**agg_dict)
    )

    first_bin = int(np.floor(
        (ASSESSMENT_START - EVENT_DATE).days
        / BIN_DAYS
    ))

    last_bin = int(np.floor(
        (ASSESSMENT_END - EVENT_DATE).days
        / BIN_DAYS
    ))

    complete = pd.DataFrame({
        "Bin_id": np.arange(
            first_bin,
            last_bin + 1,
        )
    })

    complete["Bin_start"] = (
        EVENT_DATE
        + pd.to_timedelta(
            complete["Bin_id"] * BIN_DAYS,
            unit="D",
        )
    )

    return complete.merge(
        binned,
        on="Bin_id",
        how="left",
    )


binned_groups = {
    group: build_binned_profile(
        event_groups[group],
        date_col="date",
        value_col="Pct_baseline",
        sc_col="SC_pct",
    )
    for group in GROUP_ORDER
}

ngcp_binned = build_binned_profile(
    df_ngcp_event,
    date_col="Date",
    value_col="Pct_baseline",
)

In [63]:
fig_binned = go.Figure()

for group in GROUP_ORDER:
    meta = GROUP_META[group]
    binned = binned_groups[group]

    fig_binned.add_trace(
        go.Scatter(
            x=binned["Bin_start"],
            y=binned["Median"],
            mode="lines+markers",
            connectgaps=False,
            line=dict(
                color=meta["color"],
                width=2.2,
            ),
            marker=dict(size=8),
            error_y=dict(
                type="data",
                symmetric=False,
                array=(
                    binned["P75"]
                    - binned["Median"]
                ).clip(lower=0),
                arrayminus=(
                    binned["Median"]
                    - binned["P25"]
                ).clip(lower=0),
                color=hex_to_rgba(
                    meta["color"],
                    0.65,
                ),
                thickness=1.5,
                width=3,
            ),
            customdata=np.column_stack([
                binned["N_obs"],
                binned["Median_SC"],
            ]),
            hovertemplate=(
                "Bin start: %{x|%Y-%m-%d}<br>"
                "Median: %{y:.1f}% of baseline<br>"
                "Qualified observations: "
                "%{customdata[0]:.0f}<br>"
                "Median SC: "
                "%{customdata[1]:.1f}%"
                "<extra></extra>"
            ),
            name=(
                f"{meta['code']}: "
                f"{meta['label']}"
            ),
        )
    )

fig_binned.add_trace(
    go.Scatter(
        x=ngcp_binned["Bin_start"],
        y=ngcp_binned["Median"],
        mode="lines+markers",
        connectgaps=False,
        line=dict(
            color="black",
            width=3,
        ),
        marker=dict(size=7),
        error_y=dict(
            type="data",
            symmetric=False,
            array=(
                ngcp_binned["P75"]
                - ngcp_binned["Median"]
            ).clip(lower=0),
            arrayminus=(
                ngcp_binned["Median"]
                - ngcp_binned["P25"]
            ).clip(lower=0),
            color="rgba(0,0,0,0.55)",
            thickness=1.5,
            width=3,
        ),
        name="NGCP 1 AM load",
    )
)

fig_binned.add_hline(
    y=100,
    line_dash="dot",
    line_color="grey",
    line_width=1.5,
)

add_haiyan_marker(fig_binned)

fig_binned.update_layout(
    title=(
        "Fourteen-Day Haiyan Recovery Envelopes"
        "<br><sup>"
        "Medians and temporal IQRs; "
        "empty bins are not interpolated"
        "</sup>"
    ),
    xaxis_title="Bin start date",
    yaxis_title=(
        "Percentage of pre-Haiyan baseline (%)"
    ),
)

apply_figure_style(
    fig_binned,
    width=1300,
    height=680,
    top=110,
)

fig_binned.show()

export_figure(
    fig_binned,
    OUTPUT_MAIN_FIG_DIR
    / "fig_03_haiyan_14day_recovery_envelopes_g1_g3",
)

Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/outputs/figures/main/fig_03_haiyan_14day_recovery_envelopes_g1_g3.html


In [64]:
def calculate_ntl_recovery_metrics(
    group,
    data,
):
    meta = GROUP_META[group]

    impact_window = data[
        (data["date"] >= EVENT_DATE)
        & (data["date"] <= IMPACT_END)
    ].dropna(subset=["Pct_baseline"])

    if impact_window.empty:
        return None

    impact_row = impact_window.loc[
        impact_window[
            "Pct_baseline"
        ].idxmin()
    ]

    impact_date = impact_row["date"]
    impact_pct = float(
        impact_row["Pct_baseline"]
    )

    recovery = (
        data[data["date"] >= impact_date]
        .dropna(subset=["Pct_baseline"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    result = {
        "Source": meta["code"],
        "Label": meta["label"],
        "Impact_date": impact_date,
        "Impact_pct_baseline": impact_pct,
        "Impact_drop_pct": 100 - impact_pct,
    }

    for label, fraction in [
        ("T50", 0.50),
        ("T80", 0.80),
    ]:
        target = (
            impact_pct
            + fraction * (100 - impact_pct)
        )

        candidates = recovery.index[
            recovery["Pct_baseline"] >= target
        ].tolist()

        first_crossing = (
            recovery.loc[
                candidates[0],
                "date",
            ]
            if candidates
            else pd.NaT
        )

        confirmed_index = None

        for index in candidates:
            if index + 1 >= len(recovery):
                continue

            current = recovery.loc[index]
            following = recovery.loc[index + 1]

            gap = (
                following["date"]
                - current["date"]
            ).days

            if (
                following["Pct_baseline"] >= target
                and gap <= MAX_CONFIRMATION_GAP
            ):
                confirmed_index = index
                break

        lower_date = pd.NaT
        upper_date = pd.NaT
        status = "Unconfirmed"

        if confirmed_index is not None:
            upper_date = recovery.loc[
                confirmed_index,
                "date",
            ]

            previous_below = recovery[
                (recovery.index < confirmed_index)
                & (
                    recovery["Pct_baseline"]
                    < target
                )
            ]

            lower_date = (
                previous_below.iloc[-1]["date"]
                if not previous_below.empty
                else impact_date
            )

            status = "Confirmed interval"

        result.update({
            f"{label}_target_pct": target,
            f"{label}_first_crossing": (
                first_crossing
            ),
            f"{label}_lower_date": lower_date,
            f"{label}_upper_date": upper_date,
            f"{label}_interval_days": (
                (upper_date - lower_date).days
                if (
                    pd.notna(lower_date)
                    and pd.notna(upper_date)
                )
                else np.nan
            ),
            f"{label}_status": status,
        })

    return result


def calculate_ngcp_recovery_metrics(data):
    impact_window = data[
        (data["Date"] >= EVENT_DATE)
        & (data["Date"] <= IMPACT_END)
    ].dropna(subset=["Pct_baseline"])

    impact_row = impact_window.loc[
        impact_window[
            "Pct_baseline"
        ].idxmin()
    ]

    impact_date = impact_row["Date"]
    impact_pct = float(
        impact_row["Pct_baseline"]
    )

    recovery = (
        data[data["Date"] >= impact_date]
        .set_index("Date")[["Pct_baseline"]]
        .reindex(
            pd.date_range(
                impact_date,
                ASSESSMENT_END,
                freq="D",
            )
        )
    )

    result = {
        "Source": "NGCP",
        "Label": "1 AM load",
        "Impact_date": impact_date,
        "Impact_pct_baseline": impact_pct,
        "Impact_drop_pct": 100 - impact_pct,
    }

    for label, fraction in [
        ("T50", 0.50),
        ("T80", 0.80),
    ]:
        target = (
            impact_pct
            + fraction * (100 - impact_pct)
        )

        above = (
            recovery["Pct_baseline"]
            .ge(target)
            .fillna(False)
        )

        sustained = (
            above
            .rolling(
                NGCP_SUSTAINED_DAYS,
                min_periods=NGCP_SUSTAINED_DAYS,
            )
            .sum()
            .eq(NGCP_SUSTAINED_DAYS)
        )

        if sustained.any():
            run_end = sustained.index[
                sustained
            ][0]

            sustained_date = (
                run_end
                - pd.Timedelta(
                    days=NGCP_SUSTAINED_DAYS - 1
                )
            )
        else:
            sustained_date = pd.NaT

        result.update({
            f"{label}_target_pct": target,
            f"{label}_first_crossing": (
                sustained_date
            ),
            f"{label}_lower_date": (
                sustained_date
            ),
            f"{label}_upper_date": (
                sustained_date
            ),
            f"{label}_interval_days": (
                0
                if pd.notna(sustained_date)
                else np.nan
            ),
            f"{label}_status": (
                f"{NGCP_SUSTAINED_DAYS}-day sustained"
                if pd.notna(sustained_date)
                else "Not reached"
            ),
        })

    return result

In [65]:
def calculate_ntl_recovery_metrics(
    group,
    data,
):
    meta = GROUP_META[group]

    impact_window = data[
        (data["date"] >= EVENT_DATE)
        & (data["date"] <= IMPACT_END)
    ].dropna(subset=["Pct_baseline"])

    if impact_window.empty:
        return None

    impact_row = impact_window.loc[
        impact_window[
            "Pct_baseline"
        ].idxmin()
    ]

    impact_date = impact_row["date"]
    impact_pct = float(
        impact_row["Pct_baseline"]
    )

    recovery = (
        data[data["date"] >= impact_date]
        .dropna(subset=["Pct_baseline"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    result = {
        "Source": meta["code"],
        "Label": meta["label"],
        "Impact_date": impact_date,
        "Impact_pct_baseline": impact_pct,
        "Impact_drop_pct": 100 - impact_pct,
    }

    for label, fraction in [
        ("T50", 0.50),
        ("T80", 0.80),
    ]:
        target = (
            impact_pct
            + fraction * (100 - impact_pct)
        )

        candidates = recovery.index[
            recovery["Pct_baseline"] >= target
        ].tolist()

        first_crossing = (
            recovery.loc[
                candidates[0],
                "date",
            ]
            if candidates
            else pd.NaT
        )

        confirmed_index = None

        for index in candidates:
            if index + 1 >= len(recovery):
                continue

            current = recovery.loc[index]
            following = recovery.loc[index + 1]

            gap = (
                following["date"]
                - current["date"]
            ).days

            if (
                following["Pct_baseline"] >= target
                and gap <= MAX_CONFIRMATION_GAP
            ):
                confirmed_index = index
                break

        lower_date = pd.NaT
        upper_date = pd.NaT
        status = "Unconfirmed"

        if confirmed_index is not None:
            upper_date = recovery.loc[
                confirmed_index,
                "date",
            ]

            previous_below = recovery[
                (recovery.index < confirmed_index)
                & (
                    recovery["Pct_baseline"]
                    < target
                )
            ]

            lower_date = (
                previous_below.iloc[-1]["date"]
                if not previous_below.empty
                else impact_date
            )

            status = "Confirmed interval"

        result.update({
            f"{label}_target_pct": target,
            f"{label}_first_crossing": (
                first_crossing
            ),
            f"{label}_lower_date": lower_date,
            f"{label}_upper_date": upper_date,
            f"{label}_interval_days": (
                (upper_date - lower_date).days
                if (
                    pd.notna(lower_date)
                    and pd.notna(upper_date)
                )
                else np.nan
            ),
            f"{label}_status": status,
        })

    return result


def calculate_ngcp_recovery_metrics(data):
    impact_window = data[
        (data["Date"] >= EVENT_DATE)
        & (data["Date"] <= IMPACT_END)
    ].dropna(subset=["Pct_baseline"])

    impact_row = impact_window.loc[
        impact_window[
            "Pct_baseline"
        ].idxmin()
    ]

    impact_date = impact_row["Date"]
    impact_pct = float(
        impact_row["Pct_baseline"]
    )

    recovery = (
        data[data["Date"] >= impact_date]
        .set_index("Date")[["Pct_baseline"]]
        .reindex(
            pd.date_range(
                impact_date,
                ASSESSMENT_END,
                freq="D",
            )
        )
    )

    result = {
        "Source": "NGCP",
        "Label": "1 AM load",
        "Impact_date": impact_date,
        "Impact_pct_baseline": impact_pct,
        "Impact_drop_pct": 100 - impact_pct,
    }

    for label, fraction in [
        ("T50", 0.50),
        ("T80", 0.80),
    ]:
        target = (
            impact_pct
            + fraction * (100 - impact_pct)
        )

        above = (
            recovery["Pct_baseline"]
            .ge(target)
            .fillna(False)
        )

        sustained = (
            above
            .rolling(
                NGCP_SUSTAINED_DAYS,
                min_periods=NGCP_SUSTAINED_DAYS,
            )
            .sum()
            .eq(NGCP_SUSTAINED_DAYS)
        )

        if sustained.any():
            run_end = sustained.index[
                sustained
            ][0]

            sustained_date = (
                run_end
                - pd.Timedelta(
                    days=NGCP_SUSTAINED_DAYS - 1
                )
            )
        else:
            sustained_date = pd.NaT

        result.update({
            f"{label}_target_pct": target,
            f"{label}_first_crossing": (
                sustained_date
            ),
            f"{label}_lower_date": (
                sustained_date
            ),
            f"{label}_upper_date": (
                sustained_date
            ),
            f"{label}_interval_days": (
                0
                if pd.notna(sustained_date)
                else np.nan
            ),
            f"{label}_status": (
                f"{NGCP_SUSTAINED_DAYS}-day sustained"
                if pd.notna(sustained_date)
                else "Not reached"
            ),
        })

    return result

In [66]:
recovery_rows = [
    calculate_ntl_recovery_metrics(
        group,
        event_groups[group],
    )
    for group in GROUP_ORDER
]

recovery_rows.append(
    calculate_ngcp_recovery_metrics(
        df_ngcp_event
    )
)

recovery_metrics_df = pd.DataFrame(
    recovery_rows
)

recovery_metrics_df.to_csv(
    OUTPUT_MAIN_TABLE_DIR
    / "table_02_haiyan_t50_t80_intervals.csv",
    index=False,
)

recovery_metrics_df

,Source,Label,Impact_date,Impact_pct_baseline,Impact_drop_pct,T50_target_pct,T50_first_crossing,T50_lower_date,T50_upper_date,T50_interval_days,T50_status,T80_target_pct,T80_first_crossing,T80_lower_date,T80_upper_date,T80_interval_days,T80_status
0,G1,Urban Core,2013-11-14,5.620174,94.379826,52.810087,2013-12-03,2013-12-12,2013-12-13,1,Confirmed interval,81.124035,2014-02-27,2014-04-09,2014-04-10,1,Confirmed interval
1,G2,Dense Urban,2013-11-14,7.641085,92.358915,53.820543,2013-12-03,2013-12-22,2013-12-25,3,Confirmed interval,81.528217,2014-02-05,2014-02-04,2014-02-05,1,Confirmed interval
2,G3,Core & Inner,2013-11-14,9.417516,90.582484,54.708758,2013-12-03,2013-12-12,2013-12-13,1,Confirmed interval,81.883503,2013-12-25,2014-02-04,2014-02-05,1,Confirmed interval
3,NGCP,1 AM load,2013-11-09,0.000000,100.000000,50.000000,2014-03-08,2014-03-08,2014-03-08,0,7-day sustained,80.000000,2014-04-24,2014-04-24,2014-04-24,0,7-day sustained


In [67]:
recovery_rows = [
    calculate_ntl_recovery_metrics(
        group,
        event_groups[group],
    )
    for group in GROUP_ORDER
]

recovery_rows.append(
    calculate_ngcp_recovery_metrics(
        df_ngcp_event
    )
)

recovery_metrics_df = pd.DataFrame(
    recovery_rows
)

recovery_metrics_df.to_csv(
    OUTPUT_MAIN_TABLE_DIR
    / "table_02_haiyan_t50_t80_intervals.csv",
    index=False,
)

recovery_metrics_df

,Source,Label,Impact_date,Impact_pct_baseline,Impact_drop_pct,T50_target_pct,T50_first_crossing,T50_lower_date,T50_upper_date,T50_interval_days,T50_status,T80_target_pct,T80_first_crossing,T80_lower_date,T80_upper_date,T80_interval_days,T80_status
0,G1,Urban Core,2013-11-14,5.620174,94.379826,52.810087,2013-12-03,2013-12-12,2013-12-13,1,Confirmed interval,81.124035,2014-02-27,2014-04-09,2014-04-10,1,Confirmed interval
1,G2,Dense Urban,2013-11-14,7.641085,92.358915,53.820543,2013-12-03,2013-12-22,2013-12-25,3,Confirmed interval,81.528217,2014-02-05,2014-02-04,2014-02-05,1,Confirmed interval
2,G3,Core & Inner,2013-11-14,9.417516,90.582484,54.708758,2013-12-03,2013-12-12,2013-12-13,1,Confirmed interval,81.883503,2013-12-25,2014-02-04,2014-02-05,1,Confirmed interval
3,NGCP,1 AM load,2013-11-09,0.000000,100.000000,50.000000,2014-03-08,2014-03-08,2014-03-08,0,7-day sustained,80.000000,2014-04-24,2014-04-24,2014-04-24,0,7-day sustained


In [73]:
fig_milestones = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=[
        (
            f"{GROUP_META[group]['code']}: "
            f"{GROUP_META[group]['label']}"
        )
        for group in GROUP_ORDER
    ],
)

ngcp_metrics = recovery_metrics_df[
    recovery_metrics_df["Source"] == "NGCP"
].iloc[0]

for row, group in enumerate(
    GROUP_ORDER,
    start=1,
):
    meta = GROUP_META[group]
    gdf = event_groups[group]

    metrics = recovery_metrics_df[
        recovery_metrics_df["Source"]
        == meta["code"]
    ].iloc[0]

    fig_milestones.add_trace(
        go.Scatter(
            x=df_ngcp_event["Date"],
            y=df_ngcp_event["Pct_baseline"],
            mode="lines",
            line=dict(
                color="rgba(0,0,0,0.58)",
                width=2,
            ),
            name="NGCP 1 AM load",
            legendgroup="NGCP",
            showlegend=(row == 1),
        ),
        row=row,
        col=1,
    )

    fig_milestones.add_trace(
        go.Scatter(
            x=gdf["date"],
            y=gdf["Pct_baseline"],
            mode="markers",
            marker=dict(
                size=8,
                color=meta["color"],
                line=dict(
                    color="white",
                    width=0.7,
                ),
            ),
            customdata=gdf[["SC_pct"]],
            name=f"{meta['code']} DNB-BRDF",
            legendgroup=meta["code"],
            showlegend=(row == 1),
            hovertemplate=(
                "Date: %{x|%Y-%m-%d}<br>"
                "NTL: %{y:.1f}% of baseline<br>"
                "SC: %{customdata[0]:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=row,
        col=1,
    )

    fig_milestones.add_trace(
        go.Scatter(
            x=[metrics["Impact_date"]],
            y=[metrics["Impact_pct_baseline"]],
            mode="markers",
            marker=dict(
                size=13,
                symbol="x",
                color="#1f77b4",
                line=dict(width=2),
            ),
            name="Observed impact minimum",
            legendgroup="impact",
            showlegend=(row == 1),
        ),
        row=row,
        col=1,
    )

    for label, color, dash in [
        ("T50", "#2ca02c", "dash"),
        ("T80", "#9467bd", "dot"),
    ]:
        target = metrics[
            f"{label}_target_pct"
        ]

        lower_date = metrics[
            f"{label}_lower_date"
        ]

        upper_date = metrics[
            f"{label}_upper_date"
        ]

        first_crossing = metrics[
            f"{label}_first_crossing"
        ]

        fig_milestones.add_hline(
            y=target,
            line_dash=dash,
            line_color=color,
            line_width=1.5,
            row=row,
            col=1,
        )

        fig_milestones.add_annotation(
            x=ASSESSMENT_END,
            y=target,
            text=label,
            showarrow=False,
            font=dict(
                size=12,
                color=color,
            ),
            bgcolor="rgba(255,255,255,0.80)",
            xanchor="right",
            yanchor="bottom",
            row=row,
            col=1,
        )

        if (
            pd.notna(lower_date)
            and pd.notna(upper_date)
        ):
            fig_milestones.add_vrect(
                x0=lower_date,
                x1=upper_date,
                fillcolor=hex_to_rgba(
                    color,
                    0.15,
                ),
                line_width=0,
                row=row,
                col=1,
            )

            fig_milestones.add_vline(
                x=upper_date.to_pydatetime(),
                line_dash=dash,
                line_color=color,
                line_width=2,
                row=row,
                col=1,
            )

        elif pd.notna(first_crossing):
            fig_milestones.add_trace(
                go.Scatter(
                    x=[first_crossing],
                    y=[target],
                    mode="markers",
                    marker=dict(
                        symbol="diamond-open",
                        size=12,
                        color=color,
                        line=dict(width=2),
                    ),
                    name=f"Unconfirmed {label}",
                    showlegend=(row == 1),
                ),
                row=row,
                col=1,
            )

        ngcp_date = ngcp_metrics[
            f"{label}_upper_date"
        ]

        if pd.notna(ngcp_date):
            fig_milestones.add_vline(
                x=ngcp_date.to_pydatetime(),
                line_dash="dashdot",
                line_color=color,
                line_width=1.3,
                row=row,
                col=1,
            )

    add_haiyan_marker(
        fig_milestones,
        row=row,
        col=1,
        annotation=(row == 1),
    )

    fig_milestones.update_yaxes(
        title_text="Baseline (%)",
        rangemode="tozero",
        row=row,
        col=1,
    )

fig_milestones.update_layout(
    title=(
        "Observation-Bounded T50 and T80 after Haiyan"
        "<br><sup>"
        "Shading: NTL crossing interval · "
        "dash-dot verticals: sustained NGCP crossing"
        "</sup>"
    )
)

fig_milestones.update_xaxes(
    title_text="Date",
    row=3,
    col=1,
)

apply_figure_style(
    fig_milestones,
    width=1300,
    height=800,
    top=115,
)

fig_milestones.show()

export_figure(
    fig_milestones,
    OUTPUT_MAIN_FIG_DIR
    / "fig_04_haiyan_t50_t80_intervals_g1_g3",
)

Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/outputs/figures/main/fig_04_haiyan_t50_t80_intervals_g1_g3.html


In [74]:
matched_groups = {}
validation_rows = []

for group in GROUP_ORDER:
    meta = GROUP_META[group]

    matched = pd.merge(
        event_groups[group][[
            "date",
            "Pct_baseline",
            "SC_pct",
            "Days_since_event",
        ]].rename(columns={
            "date": "Date",
            "Pct_baseline": (
                "NTL_pct_baseline"
            ),
        }),
        df_ngcp_event[[
            "Date",
            "Pct_baseline",
        ]].rename(columns={
            "Pct_baseline": (
                "NGCP_pct_baseline"
            ),
        }),
        on="Date",
        how="inner",
    ).dropna()

    matched["NTL_minus_NGCP"] = (
        matched["NTL_pct_baseline"]
        - matched["NGCP_pct_baseline"]
    )

    matched_groups[group] = matched

    validation_rows.append({
        "Group": group,
        "Code": meta["code"],
        "Label": meta["label"],
        "Matched_days": len(matched),
        "Pearson_r": matched[[
            "NTL_pct_baseline",
            "NGCP_pct_baseline",
        ]].corr(method="pearson").iloc[0, 1],
        "Spearman_r": matched[[
            "NTL_pct_baseline",
            "NGCP_pct_baseline",
        ]].corr(method="spearman").iloc[0, 1],
        "Median_absolute_difference_pct": (
            matched["NTL_minus_NGCP"]
            .abs()
            .median()
        ),
    })

validation_df = pd.DataFrame(
    validation_rows
)

validation_df.to_csv(
    OUTPUT_MAIN_TABLE_DIR
    / "table_03_haiyan_date_matched_ntl_ngcp.csv",
    index=False,
)

validation_df.round(3)

,Group,Code,Label,Matched_days,Pearson_r,Spearman_r,Median_absolute_difference_pct
0,30,G1,Urban Core,143,0.691,0.690,15.659
1,23-30,G2,Dense Urban,149,0.675,0.694,14.964
2,22-30,G3,Core & Inner,154,0.658,0.698,15.142


In [75]:
fig_matched = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.07,
    subplot_titles=[
        (
            f"{GROUP_META[group]['code']}: "
            f"{GROUP_META[group]['label']}"
        )
        for group in GROUP_ORDER
    ],
)

for col, group in enumerate(
    GROUP_ORDER,
    start=1,
):
    matched = matched_groups[group]

    fig_matched.add_trace(
        go.Scatter(
            x=matched["NGCP_pct_baseline"],
            y=matched["NTL_pct_baseline"],
            mode="markers",
            marker=dict(
                size=(
                    6
                    + matched["SC_pct"] / 12
                ),
                color=matched[
                    "Days_since_event"
                ],
                colorscale="RdBu_r",
                cmin=(
                    ASSESSMENT_START
                    - EVENT_DATE
                ).days,
                cmax=(
                    ASSESSMENT_END
                    - EVENT_DATE
                ).days,
                showscale=(col == 3),
                colorbar=(
                    dict(
                        title="Days from Haiyan",
                        x=1.02,
                    )
                    if col == 3
                    else None
                ),
                line=dict(
                    color="white",
                    width=0.7,
                ),
                opacity=0.85,
            ),
            customdata=np.column_stack([
                matched[
                    "Date"
                ].dt.strftime("%Y-%m-%d"),
                matched["SC_pct"],
            ]),
            hovertemplate=(
                "Date: %{customdata[0]}<br>"
                "NGCP: %{x:.1f}%<br>"
                "NTL: %{y:.1f}%<br>"
                "SC: %{customdata[1]:.1f}%"
                "<extra></extra>"
            ),
            showlegend=False,
        ),
        row=1,
        col=col,
    )

    axis_min = min(
        matched["NGCP_pct_baseline"].min(),
        matched["NTL_pct_baseline"].min(),
    )

    axis_max = max(
        matched["NGCP_pct_baseline"].max(),
        matched["NTL_pct_baseline"].max(),
    )

    fig_matched.add_trace(
        go.Scatter(
            x=[axis_min, axis_max],
            y=[axis_min, axis_max],
            mode="lines",
            line=dict(
                color="grey",
                dash="dot",
                width=1.5,
            ),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=col,
    )

    fig_matched.update_xaxes(
        title_text="NGCP baseline (%)",
        row=1,
        col=col,
    )

    fig_matched.update_yaxes(
        title_text=(
            "NTL baseline (%)"
            if col == 1
            else None
        ),
        row=1,
        col=col,
    )

fig_matched.update_layout(
    title=(
        "Date-Matched NTL and NGCP Conditions around Haiyan"
        "<br><sup>"
        "Marker size: spatial completeness · "
        "Colour: days from landfall"
        "</sup>"
    )
)

apply_figure_style(
    fig_matched,
    width=1400,
    height=600,
    top=115,
    right=120,
)

fig_matched.show()

export_figure(
    fig_matched,
    OUTPUT_SUPP_FIG_DIR
    / "fig_05_haiyan_ntl_ngcp_date_matched_g1_g3",
)

Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/outputs/figures/supplementary/fig_05_haiyan_ntl_ngcp_date_matched_g1_g3.html


In [76]:
fig_divergence = go.Figure()

for group in GROUP_ORDER:
    meta = GROUP_META[group]
    matched = matched_groups[group]

    fig_divergence.add_trace(
        go.Scatter(
            x=matched["Date"],
            y=matched["NTL_minus_NGCP"],
            mode="markers",
            marker=dict(
                size=(
                    6
                    + matched["SC_pct"] / 14
                ),
                color=meta["color"],
                line=dict(
                    color="white",
                    width=0.7,
                ),
                opacity=0.82,
            ),
            customdata=matched[["SC_pct"]],
            hovertemplate=(
                "Date: %{x|%Y-%m-%d}<br>"
                "NTL − NGCP: "
                "%{y:+.1f} percentage points<br>"
                "SC: %{customdata[0]:.1f}%"
                "<extra></extra>"
            ),
            name=(
                f"{meta['code']}: "
                f"{meta['label']}"
            ),
        )
    )

fig_divergence.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    line_width=1.5,
)

add_haiyan_marker(fig_divergence)

fig_divergence.update_layout(
    title=(
        "Divergence between NTL and NGCP Recovery Evidence"
        "<br><sup>"
        "Positive values indicate NTL above the "
        "contemporaneous NGCP baseline percentage"
        "</sup>"
    ),
    xaxis_title="Date",
    yaxis_title=(
        "NTL − NGCP (percentage points)"
    ),
)

apply_figure_style(
    fig_divergence,
    width=1300,
    height=650,
    top=115,
)

fig_divergence.show()

export_figure(
    fig_divergence,
    OUTPUT_SUPP_FIG_DIR
    / "fig_06_haiyan_ntl_ngcp_divergence_g1_g3",
)

Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/outputs/figures/supplementary/fig_06_haiyan_ntl_ngcp_divergence_g1_g3.html


In [77]:
SENSITIVITY_THRESHOLDS = [
    30,
    40,
    50,
    60,
    70,
    80,
]

sensitivity_rows = []

for group in GROUP_ORDER:
    meta = GROUP_META[group]

    for threshold in SENSITIVITY_THRESHOLDS:
        gdf = df_panel[
            (df_panel["Group"] == group)
            & (
                df_panel["date"]
                >= ASSESSMENT_START
            )
            & (
                df_panel["date"]
                <= ASSESSMENT_END
            )
            & (
                df_panel["SC_pct"]
                >= threshold
            )
            & df_panel["NTL_value"].notna()
        ].copy()

        baseline_values = gdf.loc[
            (gdf["date"] >= BASELINE_START)
            & (gdf["date"] <= BASELINE_END),
            "NTL_value",
        ]

        baseline = baseline_values.median()

        if (
            len(baseline_values) < 3
            or not np.isfinite(baseline)
            or baseline <= 0
        ):
            continue

        gdf["Pct_baseline"] = (
            gdf["NTL_value"]
            / baseline
            * 100
        )

        metrics = (
            calculate_ntl_recovery_metrics(
                group,
                gdf,
            )
        )

        sensitivity_rows.append({
            "Group": group,
            "Code": meta["code"],
            "Label": meta["label"],
            "Threshold": threshold,
            "Baseline_observations": len(
                baseline_values
            ),
            "Assessment_observations": len(
                gdf
            ),
            "Impact_drop_pct": (
                metrics["Impact_drop_pct"]
                if metrics is not None
                else np.nan
            ),
            "T50_upper_days": (
                (
                    metrics["T50_upper_date"]
                    - EVENT_DATE
                ).days
                if (
                    metrics is not None
                    and pd.notna(
                        metrics["T50_upper_date"]
                    )
                )
                else np.nan
            ),
            "T80_upper_days": (
                (
                    metrics["T80_upper_date"]
                    - EVENT_DATE
                ).days
                if (
                    metrics is not None
                    and pd.notna(
                        metrics["T80_upper_date"]
                    )
                )
                else np.nan
            ),
        })

sensitivity_df = pd.DataFrame(
    sensitivity_rows
)

sensitivity_df.to_csv(
    OUTPUT_SUPP_TABLE_DIR
    / "table_04_haiyan_threshold_sensitivity_g1_g3.csv",
    index=False,
)

sensitivity_df.round(2)

,Group,Code,Label,Threshold,Baseline_observations,Assessment_observations,Impact_drop_pct,T50_upper_days,T80_upper_days
0,30,G1,Urban Core,30,17,112,95.09,35,164.0
1,30,G1,Urban Core,40,15,97,95.09,35,164.0
2,30,G1,Urban Core,50,13,82,95.09,89,NaN
3,30,G1,Urban Core,60,13,75,95.09,89,NaN
4,30,G1,Urban Core,70,13,59,86.00,89,NaN
5,30,G1,Urban Core,80,7,42,66.54,147,NaN
6,23-30,G2,Dense Urban,30,18,113,92.54,47,89.0
7,23-30,G2,Dense Urban,40,16,102,92.69,78,147.0
8,23-30,G2,Dense Urban,50,11,82,92.71,78,147.0
9,23-30,G2,Dense Urban,60,11,70,92.71,78,147.0


In [78]:
fig_sensitivity = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "Qualified observations",
        "Observed impact drop",
        "Confirmed milestone upper bound",
    ),
)

for group in GROUP_ORDER:
    meta = GROUP_META[group]

    sdf = sensitivity_df[
        sensitivity_df["Group"] == group
    ]

    fig_sensitivity.add_trace(
        go.Scatter(
            x=sdf["Threshold"],
            y=sdf["Assessment_observations"],
            mode="lines+markers",
            line=dict(
                color=meta["color"],
                width=2,
            ),
            marker=dict(size=8),
            name=(
                f"{meta['code']}: "
                f"{meta['label']}"
            ),
            legendgroup=meta["code"],
        ),
        row=1,
        col=1,
    )

    fig_sensitivity.add_trace(
        go.Scatter(
            x=sdf["Threshold"],
            y=sdf["Impact_drop_pct"],
            mode="lines+markers",
            line=dict(
                color=meta["color"],
                width=2,
            ),
            marker=dict(size=8),
            showlegend=False,
            legendgroup=meta["code"],
        ),
        row=2,
        col=1,
    )

    fig_sensitivity.add_trace(
        go.Scatter(
            x=sdf["Threshold"],
            y=sdf["T50_upper_days"],
            mode="lines+markers",
            line=dict(
                color=meta["color"],
                width=2,
            ),
            marker=dict(
                size=8,
                symbol="circle",
            ),
            name=f"{meta['code']} T50",
            showlegend=False,
        ),
        row=3,
        col=1,
    )

    fig_sensitivity.add_trace(
        go.Scatter(
            x=sdf["Threshold"],
            y=sdf["T80_upper_days"],
            mode="lines+markers",
            line=dict(
                color=meta["color"],
                width=2,
                dash="dot",
            ),
            marker=dict(
                size=8,
                symbol="diamond",
            ),
            name=f"{meta['code']} T80",
            showlegend=False,
        ),
        row=3,
        col=1,
    )

for row in [1, 2, 3]:
    fig_sensitivity.add_vline(
        x=VALID_THRESHOLD,
        line_dash="dash",
        line_color="#1f77b4",
        line_width=1.8,
        row=row,
        col=1,
    )

fig_sensitivity.update_layout(
    title=(
        "Haiyan Recovery-Metric Sensitivity "
        "to Spatial Completeness"
        "<br><sup>"
        "Blue dashed line: RQ1-selected 60% threshold · "
        "dotted milestone lines: T80"
        "</sup>"
    )
)

fig_sensitivity.update_yaxes(
    title_text="Observations",
    row=1,
    col=1,
)

fig_sensitivity.update_yaxes(
    title_text="Impact drop (%)",
    row=2,
    col=1,
)

fig_sensitivity.update_yaxes(
    title_text="Days after Haiyan",
    row=3,
    col=1,
)

fig_sensitivity.update_xaxes(
    title_text=(
        "Spatial-completeness threshold (%)"
    ),
    row=3,
    col=1,
)

apply_figure_style(
    fig_sensitivity,
    width=1300,
    height=900,
    top=115,
)

fig_sensitivity.show()

export_figure(
    fig_sensitivity,
    OUTPUT_SUPP_FIG_DIR
    / "fig_07_haiyan_threshold_sensitivity_g1_g3",
)

Exported: /Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/outputs/figures/supplementary/fig_07_haiyan_threshold_sensitivity_g1_g3.html
